# 模型推理与土地覆盖地图生成

**目标**: 使用已经训练好的ViT模型，对Part 1中生成的所有64x64图像块进行分类，并将分类结果拼接成一张可视化的土地覆盖地图。

**流程**: 
1.  加载与训练时完全相同的模型结构，并载入已保存的模型权重。
2.  定义一个自定义的数据集（Dataset）来加载Part 1中生成的所有图像块。
3.  对所有图像块执行推理（Inference），得到每个块的分类标签。
4.  根据图像块的文件名中包含的坐标信息，将分类结果重新组合（Reconstruct）成一个二维地图。
5.  为不同类别分配颜色，将最终的分类地图可视化并保存。

### 步骤 1: 导入必要的库并设置环境

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import vit_b_32 # 确保使用与训练时相同的模型结构
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

### 步骤 2: 加载模型

我们将实例化一个与你训练时**完全相同**的`vit_b_32`模型结构，然后加载你保存的`.pth`权重文件。

In [ ]:
# --- 用户配置区域 ---

# 1. 模型权重文件所在的路径
models_path = "./models/"

# 2. 你训练好的模型文件名 (不含.pth后缀)
model_name = "ViT32-Base-Scratch" # <--- 请修改为你的模型文件名

# --- 配置结束 ---

# 确定设备
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"将使用设备: {DEVICE}")

# 标签映射 (必须与训练时使用的 index_to_label 保持一致)
index_to_label = {
    0: 'AnnualCrop', 1: 'Forest', 2: 'HerbaceousVegetation', 3: 'Highway', 
    4: 'Industrial', 5: 'Pasture', 6: 'PermanentCrop', 7: 'Residential', 
    8: 'River', 9: 'SeaLake'
}
num_classes = len(index_to_label)

# 实例化与训练时相同的模型结构 (vit_b_32)
model = vit_b_32()
# 基础版(Base)的 head 输入维度是 768
model.heads.head = nn.Linear(768, num_classes)

# 加载训练好的模型权重
model_save_path = os.path.join(models_path, model_name + ".pth")

try:
    model.load_state_dict(torch.load(model_save_path, map_location=DEVICE))
    print(f"成功加载模型权重: {model_save_path}")
except FileNotFoundError:
    print(f"错误: 找不到模型文件 '{model_save_path}'。请检查路径和文件名。")
except Exception as e:
    print(f"加载模型时发生错误: {e}")

model.to(DEVICE)

# ！！！关键：将模型设置为评估模式 ！！！
# 这会禁用 Dropout 等只在训练时使用的层，确保预测结果的一致性。
model.eval()


### 步骤 3: 准备数据加载器

我们需要定义一个自定义的 `Dataset` 类，它能够读取 `./patches_64x64/` 文件夹中的所有图像块，并应用与训练时完全相同的预处理变换。同时，我们还需要解析文件名来获取每个块的坐标。

In [ ]:
# 1. 定义与训练时完全相同的图像预处理流程
# 注意：因为图像块已经是64x64，所以不需要Resize或Crop
inference_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. 创建自定义数据集类
class PatchDataset(Dataset):
    def __init__(self, patch_dir, transform=None):
        self.patch_dir = patch_dir
        self.transform = transform
        # 获取所有patch的文件名，并过滤掉非png文件
        self.patch_files = sorted([f for f in os.listdir(patch_dir) if f.endswith('.png')], 
                                  key=lambda x: (int(x.split('_')[1]), int(x.split('_')[2].split('.')[0])))

    def __len__(self):
        return len(self.patch_files)

    def __getitem__(self, idx):
        img_name = self.patch_files[idx]
        img_path = os.path.join(self.patch_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        # 从文件名解析坐标
        parts = img_name.split('.')[0].split('_')
        row = int(parts[1])
        col = int(parts[2])
        
        return image, (row, col)

# 3. 实例化Dataset和DataLoader
patches_dir = "./RGB_patches_64x64/"
patch_dataset = PatchDataset(patch_dir=patches_dir, transform=inference_transform)

# 批量大小可以根据你的GPU显存调整
batch_size = 64 

patch_loader = DataLoader(patch_dataset, batch_size=batch_size, shuffle=False)

print(f"找到了 {len(patch_dataset)} 个图像块，将以 {batch_size} 的批量大小进行处理。")

### 步骤 4: 执行推理并构建分类地图

现在，我们遍历所有数据，用模型进行预测，并将结果填充到一个代表最终地图的二维数组中。

In [ ]:
# 1. 确定地图的最终尺寸
max_row = 0
max_col = 0
for filename in patch_dataset.patch_files:
    parts = filename.split('.')[0].split('_')
    max_row = max(max_row, int(parts[1]))
    max_col = max(max_col, int(parts[2]))

map_height = max_row + 1
map_width = max_col + 1
classification_map = np.zeros((map_height, map_width), dtype=np.int32)

print(f"将要生成的分类地图尺寸为: {map_height} x {map_width}")

# 2. 循环执行推理
with torch.no_grad(): # 禁用梯度计算以加速并节省内存
    for images, coords in tqdm(patch_loader, desc="正在分类"):
        images = images.to(DEVICE)
        
        # 模型预测
        outputs = model(images)
        _, predicted_indices = torch.max(outputs, 1)
        
        # 将预测结果移回CPU
        predictions_np = predicted_indices.cpu().numpy()
        
        # 将结果填充到地图上
        rows, cols = coords
        for i in range(len(predictions_np)):
            classification_map[rows[i], cols[i]] = predictions_np[i]

print("\n分类地图已在内存中生成。")

### 步骤 5: 可视化土地覆盖地图

最后一步是为地图上的每个类别索引分配一种颜色，并使用 `matplotlib` 将其显示出来。我们还会添加一个图例，以便解读地图。

In [ ]:
import matplotlib.patches as mpatches

# 1. 定义一个颜色映射 (RGB格式, 0-255)
color_map = {
    0: [255, 255, 0],    # AnnualCrop (黄色)
    1: [0, 128, 0],      # Forest (深绿)
    2: [152, 251, 152],  # HerbaceousVegetation (浅绿)
    3: [128, 128, 128],  # Highway (灰色)
    4: [139, 0, 0],      # Industrial (深红)
    5: [255, 165, 0],    # Pasture (橙色)
    6: [210, 105, 30],   # PermanentCrop (巧克力色)
    7: [255, 0, 0],      # Residential (红色)
    8: [0, 0, 255],      # River (蓝色)
    9: [0, 191, 255]     # SeaLake (深天蓝)
}

# 2. 将分类地图（整数）转换为RGB图像（三维数组）
rgb_map = np.zeros((map_height, map_width, 3), dtype=np.uint8)
for i in range(map_height):
    for j in range(map_width):
        class_index = classification_map[i, j]
        rgb_map[i, j] = color_map[class_index]

# 3. 使用matplotlib进行可视化
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(rgb_map)
ax.set_title("土地覆盖分类地图 (Land Cover Classification Map)", fontsize=16)
ax.axis('off') # 不显示坐标轴

# 4. 创建并显示图例
legend_patches = [mpatches.Patch(color=np.array(c)/255., label=l) 
                  for l, c in zip(index_to_label.values(), color_map.values())]

ax.legend(handles=legend_patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

plt.show()

# 5. (可选) 将最终地图保存为图像文件
result_image = Image.fromarray(rgb_map)
map_filename = "land_cover_map_64x64.png"
result_image.save(map_filename)
print(f"\n最终的分类地图已保存为: '{map_filename}'")